<a href="https://colab.research.google.com/github/santiagodie/face-similarity-search-pipeline/blob/main/space/build_demo_index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build the synthetic-face demo index and deploy the Space

This notebook rebuilds the public demo of [face-similarity-search-pipeline](https://github.com/santiagodie/face-similarity-search-pipeline) on **synthetic faces only** and uploads it to the Hugging Face Space `santiagodie/face-similarity-search-demo`.

What it does:

1. Gets the 750 synthetic faces in the `a small samples (750 images)` folder of [SFHQ part 3](https://www.kaggle.com/datasets/selfishgene/synthetic-faces-high-quality-sfhq-part-3) (CC0), as listed in `space/sfhq_pt3_sample_manifest.txt`. They are read from the Hugging Face dataset `santiagodie/sfhq-pt3-sample-750`; on the very first run, when that mirror does not exist yet, the notebook downloads them from Kaggle and creates it.
2. Creates 4 variations of each face (tighter crop, slight rotation, brightness change, horizontal flip), so every synthetic identity has several matches in the index.
3. Runs the original pipeline (code reused from notebooks 02-04):
   - dlib HOG detection with dlib CNN (MMOD) as a fallback;
   - crop with a 20% margin and resize to 160×160;
   - 128-d FaceNet embeddings through DeepFace;
   - a Euclidean VP-tree.
4. Saves `vptree.pkl`, the crops, and a CSV mapping each embedding to its identity and source image.
5. Creates the Space if it does not exist and uploads `space/` plus the generated files. Running the notebook again updates the Space without duplicating files.

**Before running:** add your Hugging Face write token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar), with notebook access enabled. A T4 GPU runtime is faster; a CPU runtime also works, just more slowly. Then use *Runtime → Run all*.

**First run only:** Kaggle rate-limits anonymous downloads. To create the mirror quickly, also add your Kaggle credentials as secrets: either `KAGGLE_API_TOKEN` (an API token from kaggle.com → Settings → API) or `KAGGLE_USERNAME` and `KAGGLE_KEY` (the two values in a legacy `kaggle.json`). Without them the notebook retries anonymously with long waits.

In [ ]:
# Configuration
HF_USER = "santiagodie"
SPACE_NAME = "face-similarity-search-demo"
REPO_URL = "https://github.com/santiagodie/face-similarity-search-pipeline.git"
REPO_BRANCH = "main"

SFHQ_SLUG = "selfishgene/synthetic-faces-high-quality-sfhq-part-3"
SFHQ_FOLDER = "a small samples (750 images)"
DATASET_REPO = f"{HF_USER}/sfhq-pt3-sample-750"   # CC0 mirror of the 750 faces on Hugging Face

VARIATIONS = ["orig", "crop", "rot", "bright", "flip"]  # original + 4 variations
MAX_SIDE = 600          # same downscaling as the Gradio app
EMBED_BATCH = 32
THRESHOLD = 9.655       # F1-optimal distance from notebook 05

WORK_DIR = "/content/work"
REPO_DIR = "/content/face-similarity-search-pipeline"

import time
T_START = time.time()
TIMINGS = {}

## 1. Install pinned dependencies

The versions match `space/requirements.txt`. `dlib-bin` ships prebuilt dlib wheels, so nothing is compiled.

In [ ]:
import sys
_numpy_preloaded = sys.modules.get("numpy")
_numpy_preloaded_version = getattr(_numpy_preloaded, "__version__", None)

!pip uninstall -y -q dlib
!pip install -q numpy==2.2.6 opencv-python==4.12.0.88 tensorflow==2.21.0 tf-keras==2.21.0 deepface==0.0.100 dlib-bin==20.0.1 vptree==1.3 huggingface_hub==1.33.0

import importlib.metadata as md
if _numpy_preloaded_version and _numpy_preloaded_version != md.version("numpy"):
    raise RuntimeError(
        "NumPy was already loaded with a different version. "
        "Use Runtime -> Restart session, then Runtime -> Run all again "
        "(the packages are already installed, so this is quick)."
    )
for pkg in ["numpy", "opencv-python", "tensorflow", "tf-keras", "deepface", "dlib-bin", "vptree", "huggingface_hub"]:
    print(f"{pkg:16} {md.version(pkg)}")

## 2. Get the Space files from GitHub

The Space code (`app.py`, `requirements.txt`, `packages.txt`, `README.md`, `examples/`) and the manifest live in the `space/` folder of the repository.

In [ ]:
import subprocess
from pathlib import Path

if Path(REPO_DIR, ".git").exists():
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

SPACE_DIR = Path(REPO_DIR) / "space"
CROPS_DIR = SPACE_DIR / "carasRecortadas"
MIRROR_DIR = Path(WORK_DIR) / "sfhq"
SRC_DIR = MIRROR_DIR / "images"
VAR_DIR = Path(WORK_DIR) / "variations"
for d in (SRC_DIR, VAR_DIR):
    d.mkdir(parents=True, exist_ok=True)

manifest = [l.strip() for l in (SPACE_DIR / "sfhq_pt3_sample_manifest.txt").read_text().splitlines()
            if l.strip() and not l.startswith("#")]
print(f"Manifest: {len(manifest)} faces")

## 3. Get the synthetic faces

1. Download the faces from the Hugging Face mirror `DATASET_REPO` (public, no token needed).
2. Only if the mirror does not exist yet or is incomplete: download the missing faces from Kaggle and upload them to the mirror, together with its dataset card (`space/sfhq_dataset_card.md`) and the manifest. Later runs never contact Kaggle.

In [ ]:
import base64
import urllib.error
import urllib.parse
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from google.colab import userdata
from huggingface_hub import HfApi, snapshot_download

def faltan():
    return [n for n in manifest if not (SRC_DIR / n).exists() or (SRC_DIR / n).stat().st_size == 0]

def secreto(nombre):
    try:
        return userdata.get(nombre)
    except Exception:          # secret not defined or notebook access not granted
        return None

t0 = time.time()
try:
    snapshot_download(repo_id=DATASET_REPO, repo_type="dataset",
                      allow_patterns=["images/*.jpg"], local_dir=str(MIRROR_DIR))
except Exception as e:
    print(f"Mirror {DATASET_REPO} not available yet ({type(e).__name__}).")
SRC_DIR.mkdir(parents=True, exist_ok=True)
pendientes = faltan()
FACES_SOURCE = f"Hugging Face dataset {DATASET_REPO}"
print(f"From the Hugging Face mirror: {len(manifest) - len(pendientes)}/{len(manifest)} faces")

In [ ]:
# First run only: download the missing faces from Kaggle and create or complete the mirror.
if pendientes:
    FACES_SOURCE = f"Kaggle ({len(pendientes)} files), then mirrored to {DATASET_REPO}"
    cabeceras = {"User-Agent": "Mozilla/5.0"}
    token_kaggle = secreto("KAGGLE_API_TOKEN")
    usuario_kaggle, clave_kaggle = secreto("KAGGLE_USERNAME"), secreto("KAGGLE_KEY")
    if token_kaggle:
        cabeceras["Authorization"] = f"Bearer {token_kaggle}"
        modo, hilos, espera_base = "Kaggle API token", 8, 5
    elif usuario_kaggle and clave_kaggle:
        credencial = base64.b64encode(f"{usuario_kaggle}:{clave_kaggle}".encode()).decode()
        cabeceras["Authorization"] = f"Basic {credencial}"
        modo, hilos, espera_base = "Kaggle username + key", 8, 5
    else:
        modo, hilos, espera_base = "anonymous (rate-limited, slow)", 2, 30
    print(f"Downloading {len(pendientes)} faces from Kaggle: {modo}")

    def descargar(nombre, intentos=6):
        ruta = urllib.parse.quote(f"{SFHQ_FOLDER}/{nombre}", safe="")
        url = f"https://www.kaggle.com/api/v1/datasets/download/{SFHQ_SLUG}/{ruta}"
        for intento in range(intentos):
            try:
                req = urllib.request.Request(url, headers=cabeceras)
                datos = urllib.request.urlopen(req, timeout=60).read()
                if datos[:3] != b"\xff\xd8\xff":
                    raise ValueError("not a JPEG")
                (SRC_DIR / nombre).write_bytes(datos)
                return True
            except urllib.error.HTTPError as e:
                if e.code in (401, 403) and "Authorization" in cabeceras:
                    raise RuntimeError("Kaggle rejected the credentials; check the Kaggle secrets.") from e
                # Every file in the manifest exists, so 404/429 here means rate limiting.
            except Exception:
                pass
            time.sleep(min(espera_base * 2 ** intento, 300))
        return False

    for pasada in range(1, 6):
        pendientes = faltan()
        if not pendientes:
            break
        with ThreadPoolExecutor(max_workers=hilos) as ex:
            list(ex.map(descargar, pendientes))
        print(f"  pass {pasada}: {len(manifest) - len(faltan())}/{len(manifest)} faces on disk")
        if faltan():
            time.sleep(120)
    if faltan():
        raise RuntimeError(f"{len(faltan())} faces still missing. Add Kaggle credentials as secrets "
                           "or wait a while, then run this cell again (finished downloads are kept).")

    api_hf = HfApi(token=userdata.get("HF_TOKEN"))
    api_hf.create_repo(repo_id=DATASET_REPO, repo_type="dataset", private=False, exist_ok=True)
    api_hf.upload_folder(folder_path=str(SRC_DIR), path_in_repo="images", repo_id=DATASET_REPO,
                         repo_type="dataset", allow_patterns=["*.jpg"],
                         commit_message="Add SFHQ part 3 750-image sample (CC0)")
    for local, remoto in [("sfhq_dataset_card.md", "README.md"),
                          ("sfhq_pt3_sample_manifest.txt", "sfhq_pt3_sample_manifest.txt")]:
        api_hf.upload_file(path_or_fileobj=str(SPACE_DIR / local), path_in_repo=remoto,
                           repo_id=DATASET_REPO, repo_type="dataset",
                           commit_message=f"Add {remoto}")
    print(f"Mirror ready: https://huggingface.co/datasets/{DATASET_REPO}")

assert not faltan()
print(f"All {len(manifest)} faces available. Source: {FACES_SOURCE}")
TIMINGS["faces"] = time.time() - t0

## 4. Create the variations

Each face is saved as the original plus 4 variations. The parameters are deterministic, so every run produces the same images.

In [ ]:
import cv2
import numpy as np

def generar_variacion(img, variacion, i):
    h, w = img.shape[:2]
    if variacion == "orig":
        return img
    if variacion == "crop":       # tighter crop: remove 8% on each side
        m = int(0.08 * min(h, w))
        return cv2.resize(img[m:h - m, m:w - m], (w, h), interpolation=cv2.INTER_AREA)
    if variacion == "rot":        # slight rotation, alternating direction
        angulo = 8 if i % 2 == 0 else -8
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angulo, 1.0)
        return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    if variacion == "bright":     # brighter or darker, alternating
        factor = 1.3 if i % 2 == 0 else 0.75
        return cv2.convertScaleAbs(img, alpha=factor, beta=0)
    if variacion == "flip":
        return cv2.flip(img, 1)
    raise ValueError(variacion)

t0 = time.time()
trabajos = []
for i, nombre in enumerate(manifest):
    img = cv2.imread(str(SRC_DIR / nombre))
    if img is None:
        raise RuntimeError(f"Cannot read {nombre}")
    stem = Path(nombre).stem
    for variacion in VARIATIONS:
        destino = VAR_DIR / f"{stem}__{variacion}.jpg"
        if not destino.exists():
            cv2.imwrite(str(destino), generar_variacion(img, variacion, i), [cv2.IMWRITE_JPEG_QUALITY, 95])
        trabajos.append((str(destino), str(CROPS_DIR / destino.name)))
print(f"{len(manifest)} faces x {len(VARIATIONS)} versions = {len(trabajos)} images")
TIMINGS["variations"] = time.time() - t0

## 5. Detect, crop and normalise faces (notebooks 02 and 04)

Same functions as the Gradio app: downscale to at most 600 px, HOG detection with the CNN (MMOD) detector as a fallback, 20% margin, 160×160 resize. Each image contains one synthetic face, so only the largest detection is kept.

As in notebook 02, every worker process loads its own copy of the dlib models. This runs before TensorFlow is imported, so forking is safe.

In [ ]:
import bz2
import os
import shutil
import multiprocessing
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
import dlib

CNN_PATH = Path(WORK_DIR) / "mmod_human_face_detector.dat"
if not CNN_PATH.exists():
    urllib.request.urlretrieve(
        "https://github.com/davisking/dlib-models/raw/master/mmod_human_face_detector.dat.bz2",
        str(CNN_PATH) + ".bz2")
    with bz2.open(str(CNN_PATH) + ".bz2") as src, open(CNN_PATH, "wb") as dst:
        dst.write(src.read())

# auxiliary function from PyImageSearch helpers.py (notebook 02)
def convert_and_trim_bb(image, rect):
    startX = max(0, rect.left())
    startY = max(0, rect.top())
    endX   = min(rect.right(),  image.shape[1])
    endY   = min(rect.bottom(), image.shape[0])
    return (startX, startY, endX - startX, endY - startY)

def recortarYNormalizar(imagen_bgr, x, y, w, h):
    ih, iw      = imagen_bgr.shape[:2]
    margen_px_w = int(w * 0.20)
    margen_px_h = int(h * 0.20)
    x1 = max(0,  x - margen_px_w)
    y1 = max(0,  y - margen_px_h)
    x2 = min(iw, x + w + margen_px_w)
    y2 = min(ih, y + h + margen_px_h)
    if (x2 - x1) < 20 or (y2 - y1) < 20:
        return None
    return cv2.resize(imagen_bgr[y1:y2, x1:x2], (160, 160),
                      interpolation=cv2.INTER_CUBIC)

_detectorHOG = None
_detectorCNN = None

def _init_detectores(cnn_path):
    global _detectorHOG, _detectorCNN
    _detectorHOG = dlib.get_frontal_face_detector()
    _detectorCNN = dlib.cnn_face_detection_model_v1(cnn_path)

def detectarCaras(imagen_bgr):
    rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    rects_hog = _detectorHOG(rgb, 1)
    if len(rects_hog) > 0:
        return [convert_and_trim_bb(imagen_bgr, r) for r in rects_hog], "HOG"
    results_cnn = _detectorCNN(rgb, 1)
    return [convert_and_trim_bb(imagen_bgr, r.rect) for r in results_cnn], "CNN"

def procesar_imagen(trabajo):
    origen, destino = trabajo
    img = cv2.imread(origen)
    if img is None:
        return "error", None
    h, w = img.shape[:2]
    if max(h, w) > MAX_SIDE:
        escala = MAX_SIDE / max(h, w)
        img = cv2.resize(img, (int(w * escala), int(h * escala)))
    caras, detector = detectarCaras(img)
    if not caras:
        return "no_face", detector
    x, y, w_, h_ = max(caras, key=lambda c: c[2] * c[3])
    recorte = recortarYNormalizar(img, x, y, w_, h_)
    if recorte is None:
        return "too_small", detector
    cv2.imwrite(destino, recorte, [cv2.IMWRITE_JPEG_QUALITY, 95])
    return "ok", detector

if CROPS_DIR.exists():
    shutil.rmtree(CROPS_DIR)   # rebuild from scratch on every run
CROPS_DIR.mkdir(parents=True)

t0 = time.time()
n_procesos = max(1, os.cpu_count() or 1)
print(f"Detecting faces in {len(trabajos)} images with {n_procesos} processes...")
resultados = []
with ProcessPoolExecutor(max_workers=n_procesos, mp_context=multiprocessing.get_context("fork"),
                         initializer=_init_detectores, initargs=(str(CNN_PATH),)) as ex:
    for k, res in enumerate(ex.map(procesar_imagen, trabajos, chunksize=8)):
        resultados.append(res)
        if k % 500 == 0:
            print(f"  {k}/{len(trabajos)}")
estados = Counter(r[0] for r in resultados)
detectores = Counter(r[1] for r in resultados if r[0] == "ok")
print(f"Crops: {estados['ok']}  no face: {estados['no_face']}  too small: {estados['too_small']}  errors: {estados['error']}")
print(f"Detector used: {dict(detectores)}")
TIMINGS["detection"] = time.time() - t0

## 6. FaceNet embeddings (notebook 03)

Same call as notebook 03 and the app: `DeepFace.represent(..., model_name="Facenet", enforce_detection=False)`, here in batches.

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
print("GPU:" if gpus else "No GPU, running on CPU", [g.name for g in gpus])

from deepface import DeepFace
DeepFace.build_model("Facenet")

nombres = sorted(p.name for p in CROPS_DIR.glob("*.jpg"))
t0 = time.time()
vecs = []
for i in range(0, len(nombres), EMBED_BATCH):
    lote = [cv2.imread(str(CROPS_DIR / n)) for n in nombres[i:i + EMBED_BATCH]]
    res = DeepFace.represent(lote, model_name="Facenet", enforce_detection=False)
    if res and isinstance(res[0], dict):
        res = [res]
    vecs += [np.array(r[0]["embedding"], dtype=np.float32) for r in res]
    if (i // EMBED_BATCH) % 20 == 0:
        print(f"  {i + len(lote)}/{len(nombres)}")
vecs_arr = np.stack(vecs)
assert vecs_arr.shape == (len(nombres), 128), vecs_arr.shape
print(f"Embeddings: {vecs_arr.shape}")
TIMINGS["embeddings"] = time.time() - t0

## 7. VP-tree and a quick check (notebooks 04 and 05)

Same `Vec` wrapper and Euclidean distance as the app. As a sanity check, each original face is used as a query: the first result should be itself, and the other results should be its own variations within the threshold.

In [ ]:
import vptree

class Vec:
    def __init__(self, idx, arr):
        self.idx = idx
        self.arr = arr
    def __lt__(self, other): return self.idx <  other.idx
    def __le__(self, other): return self.idx <= other.idx
    def __eq__(self, other): return self.idx == other.idx
    def __ge__(self, other): return self.idx >= other.idx
    def __gt__(self, other): return self.idx >  other.idx

def distancia_l2(a, b):
    return float(np.linalg.norm(a.arr - b.arr))

identidades = [n.split("__")[0] for n in nombres]
variaciones = [n.split("__")[1].rsplit(".", 1)[0] for n in nombres]

t0 = time.time()
vp = vptree.VPTree([Vec(i, vecs_arr[i]) for i in range(len(nombres))], distancia_l2)
TIMINGS["vptree"] = time.time() - t0

por_identidad = Counter(identidades)
consultas = [i for i, v in enumerate(variaciones) if v == "orig"]
self_ok = encontradas = esperadas = 0
for i in consultas:
    vecinos = vp.get_n_nearest_neighbors(Vec(-1, vecs_arr[i]), len(VARIATIONS))
    self_ok += vecinos[0][1].idx == i
    otros = [v.idx for d, v in vecinos if v.idx != i and d <= THRESHOLD]
    encontradas += sum(identidades[j] == identidades[i] for j in otros)
    esperadas += por_identidad[identidades[i]] - 1
print(f"Self-retrieval: {self_ok}/{len(consultas)}")
print(f"Variations found in the top {len(VARIATIONS)} within d <= {THRESHOLD}: {encontradas}/{esperadas} "
      f"({100 * encontradas / max(esperadas, 1):.1f}%)")

## 8. Save the index and the mapping

`vptree.pkl` keeps the format of the original notebooks (`paths`, `vecs_arr`) plus the identity of each embedding. `index_mapping.csv` maps every embedding to its identity, variation, crop and source image.

In [ ]:
import csv
import pickle

with open(SPACE_DIR / "vptree.pkl", "wb") as f:
    pickle.dump({"paths": nombres, "vecs_arr": vecs_arr, "identities": identidades}, f)

with open(SPACE_DIR / "index_mapping.csv", "w", newline="", encoding="utf-8") as f:
    escritor = csv.writer(f)
    escritor.writerow(["index", "identity", "variation", "crop_file", "source_image"])
    for i, n in enumerate(nombres):
        escritor.writerow([i, identidades[i], variaciones[i], f"carasRecortadas/{n}",
                           f"{SFHQ_FOLDER}/{identidades[i]}.jpg"])
print("Saved vptree.pkl and index_mapping.csv")

## 9. Create or update the Space

The token is read from the Colab secret `HF_TOKEN` and is never printed or written to disk. `delete_patterns` removes old crops and examples in the same commit, so a second run replaces the index instead of adding to it.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

repo_id = f"{HF_USER}/{SPACE_NAME}"
api = HfApi(token=userdata.get("HF_TOKEN"))
api.create_repo(repo_id=repo_id, repo_type="space", space_sdk="gradio", exist_ok=True)

t0 = time.time()
api.upload_folder(
    folder_path=str(SPACE_DIR),
    repo_id=repo_id,
    repo_type="space",
    ignore_patterns=["*.ipynb", ".ipynb_checkpoints/*", "__pycache__/*", "sfhq_dataset_card.md"],
    delete_patterns=["carasRecortadas/*", "examples/*"],
    commit_message="Update synthetic-face demo index",
)
TIMINGS["upload"] = time.time() - t0
SPACE_URL = f"https://huggingface.co/spaces/{repo_id}"
print(f"Uploaded to {SPACE_URL}")

## 10. Summary

In [ ]:
total = time.time() - T_START
print(f"Synthetic faces:      {len(manifest)} (source: {FACES_SOURCE})")
print(f"Versions per face:    {len(VARIATIONS)} (original + {len(VARIATIONS) - 1} variations: {', '.join(VARIATIONS[1:])})")
print(f"Images processed:     {len(trabajos)}")
print(f"Faces indexed:        {len(nombres)} ({estados['no_face'] + estados['too_small'] + estados['error']} images without a usable face)")
print(f"Hardware:             {'GPU ' + gpus[0].name if gpus else 'CPU only'}")
for etapa, segundos in TIMINGS.items():
    print(f"  {etapa:<12} {segundos / 60:6.1f} min")
print(f"Total time:           {total / 60:.1f} min")
print(f"Space:                {SPACE_URL}")
print("The Space rebuilds automatically after the upload; the first start takes a few minutes.")